# Shape-Preserving Creative 2+6 Generation (RL Edition)

A streamlined variant of `GM_demo_notebook.ipynb`. Two focus areas:

1. **H1 persistence as a *shape-realness* signal, not a novelty signal.** The earlier notebook treated large Wasserstein distance to training H1 diagrams as "novel"; here we flip it. We compute H1 persistence for class-2 and class-6 training images *separately* and score a generated image by how close it sits to **both** training classes in topological space:

   $$\text{H1Shape}(x) = \min\!\left(\frac{1}{1 + d_2(x)},\ \frac{1}{1 + d_6(x)}\right)$$

   where $d_k(x)$ is the minimum Wasserstein-1 distance between $x$'s H1 diagram and the H1 diagrams of training class $k$. Using `min` forces the generated shape to have topological features compatible with *both* 2s **and** 6s — analogous to how the original Ramaswamy usefulness term uses $\min(P_2, P_6)$ at the classifier level.

   Creativity comes "for free" from the constraint: no training digit is both a 2 and a 6, so anything scoring well under H1Shape must live in the topological overlap — a region the training distribution itself doesn't populate.

2. **RL latent-space search.** The H1 / Wasserstein term is *not differentiable*, so gradient descent can't use it directly. REINFORCE / sNES treats the reward as a black box: maintain a diagonal Gaussian policy over the VAE latent and update $(\mu, \sigma)$ with the score-function gradient, plus a KL trust region back to $\mathcal{N}(0, I)$ to keep samples on the VAE manifold.


In [ ]:
from typing import Union

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

try:
    import gudhi
    from persim import wasserstein as _wasserstein
    TOPO_AVAILABLE = True
except ImportError:
    TOPO_AVAILABLE = False
    print('gudhi / persim not installed — H1 shape term will be 0.')
    print('Install with:  pip install gudhi persim')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Data and models

- `get_mnist_26_labeled` — MNIST restricted to digits 2 and 6, returning original labels (needed to split H1 diagrams by class).
- `get_mnist_all` — full MNIST, for training the 10-class evaluator and the general AE.
- `VAE` — unconditional VAE over 2s and 6s; provides the latent space RL searches in.
- `Classifier10` — 10-class MNIST classifier, used *only* for evaluation (balanced-mass ambiguity).
- `GeneralAE` — AE trained on all 10 MNIST classes; its reconstruction error gives the pixel-level realness signal.


In [ ]:
def get_mnist_26(batch_size: int = 128, train: bool = True):
    dataset = torchvision.datasets.MNIST(
        root='./data', train=train, download=True,
        transform=transforms.ToTensor()
    )
    idx = [i for i, y in enumerate(dataset.targets.tolist()) if y in (2, 6)]
    return DataLoader(Subset(dataset, idx), batch_size=batch_size, shuffle=train)


def get_mnist_all(batch_size: int = 256, train: bool = True):
    dataset = torchvision.datasets.MNIST(
        root='./data', train=train, download=True,
        transform=transforms.ToTensor()
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=train)


def cache_flat_images_by_class(loader) -> dict:
    """Return {label: (N_k, 784) array} for every label in the loader."""
    buckets: dict = {}
    for x, y in loader:
        flat = x.view(x.size(0), -1).numpy()
        for lab in y.unique().tolist():
            mask = (y == lab).numpy()
            buckets.setdefault(int(lab), []).append(flat[mask])
    return {lab: np.concatenate(parts, axis=0) for lab, parts in buckets.items()}


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim: int = 32):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder_body = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512),        nn.ReLU(),
            nn.Linear(512, 784),        nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def encode(self, x):
        h = self.encoder_body(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def sample(self, n: int, device) -> torch.Tensor:
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decoder(z)


def vae_loss(x_recon, x, mu, logvar, beta: float = 1.0):
    recon = F.binary_cross_entropy(x_recon, x, reduction='sum') / x.size(0)
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl


class Classifier10(nn.Module):
    """10-class MNIST classifier. P(2) and P(6) come from the full softmax, so
    blurry garbage that a 2-class head would map to (0.5, 0.5) now scores low
    because the 10-class head allocates mass to 3/5/8."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

    def probs(self, x):
        return F.softmax(self.forward(x), dim=1)


class GeneralAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


In [ ]:
def train_vae(vae, loader, epochs: int = 50, lr: float = 1e-3,
              device: Union[str, torch.device] = 'cpu', beta: float = 1.0):
    opt = optim.Adam(vae.parameters(), lr=lr)
    vae.train()
    for epoch in range(epochs):
        total = 0.0
        for x, _ in loader:
            x = x.to(device)
            x_recon, mu, logvar = vae(x)
            loss = vae_loss(x_recon, x, mu, logvar, beta=beta)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f'  [VAE] epoch {epoch+1:>3}/{epochs}  loss={total/len(loader):.4f}')


def train_clf10(clf, loader, epochs: int = 5, lr: float = 1e-3,
                device: Union[str, torch.device] = 'cpu'):
    opt = optim.Adam(clf.parameters(), lr=lr)
    clf.train()
    for epoch in range(epochs):
        total, correct, n = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = clf(x)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            total   += loss.item()
            correct += (logits.argmax(1) == y).sum().item()
            n       += y.size(0)
        print(f'  [CLF10] epoch {epoch+1:>2}/{epochs}  '
              f'loss={total/len(loader):.4f}  acc={correct/n:.4f}')


def train_general_ae(ae, loader, epochs: int = 20, lr: float = 1e-3,
                     device: Union[str, torch.device] = 'cpu'):
    opt = optim.Adam(ae.parameters(), lr=lr)
    ae.train()
    for epoch in range(epochs):
        total = 0.0
        for x, _ in loader:
            x = x.to(device)
            loss = F.mse_loss(ae(x), x)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f'  [AE] epoch {epoch+1:>3}/{epochs}  loss={total/len(loader):.6f}')


## Shape-preserving creativity metrics

Three scores, combined as a weighted geometric mean:

$$\text{Creativity}(x) = \text{H1Shape}(x)^{w_1} \cdot \text{Realness}(x)^{w_2} \cdot \text{Ambiguity}(x)^{w_3}$$

- **H1Shape** (topological realness w.r.t. *both* classes) — compute the H1 persistence diagram of $x$ via a cubical sublevel-set filtration. Let $d_2(x)$ and $d_6(x)$ be the min Wasserstein-1 distances to cached class-2 and class-6 training diagrams. Then $\text{H1Shape}(x) = \min\!\big(1/(1{+}d_2),\ 1/(1{+}d_6)\big)$. Peaks only when both classes' topologies are simultaneously nearby — forces loops/strokes from both 2s and 6s.
- **Realness** (pixel-level) — $1 / (1 + \text{MSE}(x, \text{AE}(x)))$ with AE trained on all ten digits. Complements H1Shape: H1 can be spoofed by pathological pixel patterns with the right topology, and AE reconstruction catches that.
- **Ambiguity** (balanced-mass) — $(P_2 + P_6) \cdot (1 - |P_2 - P_6|/(P_2 + P_6))$ from the 10-class classifier. Penalises both "not a digit" and "one-sided."

Where did *novelty* go? It's implicit. Training images are never both strongly 2-like **and** strongly 6-like topologically — the two classes are topologically disjoint in the training set. Any image scoring high on H1Shape is therefore in a region unoccupied by training data, so novelty is a consequence of the constraint rather than a separate term.


In [ ]:
def _persistence_h1(img_np: np.ndarray) -> np.ndarray:
    """H1 persistence diagram via cubical sublevel-set filtration."""
    cc = gudhi.CubicalComplex(
        dimensions=[28, 28],
        top_dimensional_cells=img_np.flatten().tolist()
    )
    cc.compute_persistence()
    dgm = cc.persistence_intervals_in_dimension(1)
    if len(dgm) == 0:
        return np.array([[0.0, 0.0]])
    finite = dgm[np.isfinite(dgm[:, 1])]
    return finite if len(finite) > 0 else np.array([[0.0, 0.0]])


def compute_class_diagrams(flat_by_class: dict,
                            classes: tuple = (2, 6),
                            n_per_class: int = 60,
                            seed: int = 0) -> dict:
    """Compute H1 diagrams separately for each requested class.

    Returns {class_label: [diagram, ...]}. Total work is
    len(classes) * n_per_class diagrams.
    """
    rng = np.random.default_rng(seed)
    out: dict = {}
    for k in classes:
        flat_k = flat_by_class[k]
        idx    = rng.choice(len(flat_k), size=min(n_per_class, len(flat_k)),
                            replace=False)
        diagrams = []
        for i, j in enumerate(idx):
            diagrams.append(_persistence_h1(flat_k[j].reshape(28, 28)))
            if (i + 1) % 30 == 0:
                print(f'  [TOPO] class {k}: {i+1}/{len(idx)} diagrams')
        out[k] = diagrams
    return out


def _wd(d1, d2) -> float:
    r = _wasserstein(d1, d2, matching=False)
    return float(r[0]) if isinstance(r, tuple) else float(r)


def h1_shape_realness(images: torch.Tensor,
                      class_diagrams: dict) -> tuple:
    """H1-based shape realness w.r.t. both 2s and 6s.

    For each image, compute min Wasserstein-1 distance to each class's
    training diagrams, convert to closeness = 1 / (1 + d), and return
    the MIN across classes.  High only when the shape is topologically
    close to *both* class 2 and class 6.

    Returns
    -------
    h1_shape  : (N,)  min closeness across classes — the reward term
    close_per : dict {class: (N,)}  individual class closeness — diagnostic
    """
    if not TOPO_AVAILABLE or not class_diagrams:
        zeros = np.zeros(len(images), dtype=np.float32)
        return zeros, {}
    imgs_np = images.squeeze().cpu().numpy()
    if imgs_np.ndim == 2:
        imgs_np = imgs_np[None]

    close_per = {k: np.empty(len(imgs_np), dtype=np.float32)
                  for k in class_diagrams}
    for i, img in enumerate(imgs_np):
        dgm = _persistence_h1(img)
        for k, diagrams_k in class_diagrams.items():
            min_d = min(_wd(dgm, td) for td in diagrams_k)
            close_per[k][i] = 1.0 / (1.0 + min_d)

    stacked  = np.stack([close_per[k] for k in class_diagrams], axis=0)
    h1_shape = stacked.min(axis=0).astype(np.float32)
    return h1_shape, close_per


In [ ]:
def balanced_ambiguity(probs10: np.ndarray) -> np.ndarray:
    """10-class ambiguity between 2 and 6.

    (p2 + p6) * (1 - |p2 - p6| / (p2 + p6))
    First factor: is it a 2 or 6 at all?  Second: balanced between them?
    """
    p2, p6 = probs10[:, 2], probs10[:, 6]
    mass   = p2 + p6
    diff   = np.abs(p2 - p6)
    balance = 1.0 - diff / (mass + 1e-10)
    return np.clip(mass * balance, 0.0, 1.0)


def score_shape_creativity(images: torch.Tensor,
                           clf: Classifier10,
                           ae: GeneralAE,
                           class_diagrams: dict,
                           device: Union[str, torch.device] = 'cpu',
                           weights: tuple = (1/3, 1/3, 1/3)):
    """Return (creativity, h1_shape, realness, ambiguity, close_per)

    close_per is a diagnostic dict {2: closeness_to_2, 6: closeness_to_6};
    the scalar h1_shape = min over classes of those closenesses.
    """
    w1, w2, w3 = weights
    clf.eval(); ae.eval()
    with torch.no_grad():
        x       = images.to(device)
        mse     = F.mse_loss(ae(x), x, reduction='none').mean(dim=[1,2,3]).cpu().numpy()
        probs10 = clf.probs(x).cpu().numpy()
    realness                = 1.0 / (1.0 + mse)
    ambiguity               = balanced_ambiguity(probs10)
    h1_shape, close_per     = h1_shape_realness(images.cpu(), class_diagrams)
    creativity = (h1_shape ** w1) * (realness ** w2) * (ambiguity ** w3)
    return creativity, h1_shape, realness, ambiguity, close_per


## RL latent-space search (REINFORCE / sNES)

Policy: $\pi_\theta(z) = \mathcal{N}(\mu,\,\mathrm{diag}(\sigma^2))$ over $\mathbb{R}^d$.

Each episode draws $N$ latent codes, decodes them, and evaluates the *exact* shape-preserving reward (including the non-differentiable H1 Wasserstein term) then updates $(\mu, \log\sigma)$ with the score-function gradient:

$$\nabla_\mu J \approx \frac{1}{N}\sum_i (R_i - b)\,\frac{\varepsilon_i}{\sigma}, \qquad \nabla_{\log\sigma} J \approx \frac{1}{N}\sum_i (R_i - b)\,(\varepsilon_i^2 - 1)$$

with $\varepsilon_i = (z_i - \mu)/\sigma$ and an EMA baseline $b$ (Williams 1992). For diagonal Gaussian policies this is equivalent to **sNES** (Wierstra et al. 2014) — same family as the CMA-ES latent-space controller in *World Models* (Ha & Schmidhuber 2018).

A KL trust region back to $\mathcal{N}(0, I)$ (coefficient `beta`) keeps the policy inside the VAE training envelope so the decoder continues to produce digit-like marks. The policy mean is warm-started to the midpoint of the class-2 and class-6 latent centroids, placing initial sampling in the ambiguous region from the start.

H1 persistence is evaluated on every sample, so we keep the population at 32 and run ~80 episodes. Compute scales as `n_samples × n_episodes × (diagrams_per_class × n_classes)` Wasserstein calls.


In [ ]:
def _warm_start_mu(vae, loader, device, n_pairs: int = 32) -> np.ndarray:
    vae.eval()
    mus_2, mus_6 = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            m, _ = vae.encode(x)
            if (y == 2).any():
                mus_2.append(m[y == 2].cpu().numpy())
            if (y == 6).any():
                mus_6.append(m[y == 6].cpu().numpy())
            if (sum(a.shape[0] for a in mus_2) >= n_pairs
                    and sum(a.shape[0] for a in mus_6) >= n_pairs):
                break
    mu2 = np.concatenate(mus_2, axis=0)[:n_pairs].mean(axis=0)
    mu6 = np.concatenate(mus_6, axis=0)[:n_pairs].mean(axis=0)
    return (0.5 * (mu2 + mu6)).astype(np.float64)


def generate_shape_creative_rl(
    vae: VAE,
    clf: Classifier10,
    ae: GeneralAE,
    class_diagrams: dict,
    warm_start_loader,
    n_samples: int = 32,
    n_episodes: int = 80,
    lr_mu: float = 0.05,
    lr_sigma: float = 0.01,
    init_sigma: float = 1.0,
    beta: float = 0.01,
    weights: tuple = (1/3, 1/3, 1/3),
    baseline_decay: float = 0.9,
    device: Union[str, torch.device] = 'cpu',
    warm_start_pairs: int = 32,
    seed: int = 0,
) -> tuple:
    """Shape-preserving REINFORCE / sNES latent-space search.

    Reward per sample z:
        R(z) = h1_shape(decode(z))^w1 * realness^w2 * ambiguity^w3

    h1_shape = min over classes of 1/(1 + d_k) where d_k is the min
    Wasserstein-1 distance to class-k training H1 diagrams.  Non-differentiable
    — RL handles it as an opaque black-box reward.
    """
    assert TOPO_AVAILABLE, 'Install gudhi + persim for shape-preserving RL.'
    vae.eval(); clf.eval(); ae.eval()
    d = vae.latent_dim
    w1, w2, w3 = weights
    rng = np.random.default_rng(seed)

    mu = _warm_start_mu(vae, warm_start_loader, device, warm_start_pairs)
    print(f'  [RL] warm-start mu from 2/6 centroid midpoint  ||mu||={np.linalg.norm(mu):.3f}')
    log_sig  = np.full(d, np.log(init_sigma), dtype=np.float64)
    baseline = 0.0

    hist = {
        'reward': [], 'h1_shape': [], 'close_2': [], 'close_6': [],
        'realness': [], 'ambiguity': [], 'sigma': [],
    }

    for ep in range(n_episodes):
        sigma = np.exp(log_sig)
        eps   = rng.standard_normal((n_samples, d))
        z_np  = (mu[None] + sigma[None] * eps).astype(np.float32)

        with torch.no_grad():
            imgs_dev = vae.decoder(torch.tensor(z_np, device=device))
            mse      = F.mse_loss(ae(imgs_dev), imgs_dev, reduction='none') \
                        .mean(dim=[1,2,3]).cpu().numpy()
            probs10  = clf.probs(imgs_dev).cpu().numpy()
            imgs_cpu = imgs_dev.cpu()

        realness            = 1.0 / (1.0 + mse)
        ambiguity           = balanced_ambiguity(probs10)
        h1_shape, close_per = h1_shape_realness(imgs_cpu, class_diagrams)

        rewards  = (h1_shape ** w1) * (realness ** w2) * (ambiguity ** w3)
        adv      = rewards - baseline
        baseline = baseline_decay * baseline + (1 - baseline_decay) * float(rewards.mean())

        adv_col      = adv[:, None]
        grad_mu      = (adv_col * (eps / sigma[None])).mean(axis=0)
        grad_log_sig = (adv_col * (eps ** 2 - 1.0)).mean(axis=0)

        # KL trust region to N(0, I):  dKL/dmu = mu,  dKL/dlog_sig = sigma^2 - 1
        mu      += lr_mu    * (grad_mu      - beta * mu)
        log_sig += lr_sigma * (grad_log_sig - beta * (sigma**2 - 1.0))

        hist['reward'].append(float(rewards.mean()))
        hist['h1_shape'].append(float(h1_shape.mean()))
        hist['close_2'].append(float(close_per.get(2, np.zeros(1)).mean()))
        hist['close_6'].append(float(close_per.get(6, np.zeros(1)).mean()))
        hist['realness'].append(float(realness.mean()))
        hist['ambiguity'].append(float(ambiguity.mean()))
        hist['sigma'].append(float(sigma.mean()))

        if (ep + 1) % 10 == 0:
            c2 = close_per.get(2, np.zeros(1)).mean()
            c6 = close_per.get(6, np.zeros(1)).mean()
            print(f'  [RL] ep {ep+1:>3}/{n_episodes}  '
                  f'R={rewards.mean():.4f}  '
                  f'H1={h1_shape.mean():.4f}  '
                  f'(c2={c2:.3f} c6={c6:.3f})  '
                  f'real={realness.mean():.4f}  '
                  f'amb={ambiguity.mean():.4f}  '
                  f'sigma={sigma.mean():.3f}')

    # Final sample from converged policy
    sigma  = np.exp(log_sig)
    z_fin  = (mu[None] + sigma[None] * rng.standard_normal((n_samples, d))).astype(np.float32)
    with torch.no_grad():
        imgs_final = vae.decoder(torch.tensor(z_fin, device=device)).cpu()
    return imgs_final, hist


## Train models and cache per-class training diagrams

One-time setup: train the VAE on 2s and 6s, the 10-class evaluator on all MNIST, the general AE on all MNIST, and compute H1 diagrams **separately for class 2 and class 6** — the new H1Shape term needs both.


In [ ]:
torch.manual_seed(0); np.random.seed(0)

print('[1] Loading MNIST digits 2 & 6 ...')
loader_26       = get_mnist_26(batch_size=128, train=True)
flat_by_class_26 = cache_flat_images_by_class(loader_26)
for k in (2, 6):
    print(f'    class {k}: {len(flat_by_class_26[k])} training images')

print('[2] Loading full MNIST ...')
loader_all = get_mnist_all(batch_size=256, train=True)

print('[3] Training VAE on 2s and 6s ...')
vae = VAE(latent_dim=32).to(device)
train_vae(vae, loader_26, epochs=500, lr=1e-3, device=device, beta=1.0)

print('\n[4] Training 10-class classifier ...')
clf = Classifier10().to(device)
train_clf10(clf, loader_all, epochs=5, lr=1e-3, device=device)

print('\n[5] Training general MNIST AE ...')
ae = GeneralAE(latent_dim=64).to(device)
train_general_ae(ae, loader_all, epochs=50, lr=1e-3, device=device)

print('\n[6] Computing per-class H1 persistence diagrams ...')
if TOPO_AVAILABLE:
    class_diagrams = compute_class_diagrams(flat_by_class_26,
                                             classes=(2, 6),
                                             n_per_class=60, seed=0)
    for k, ds in class_diagrams.items():
        print(f'    class {k}: {len(ds)} H1 diagrams cached.')
else:
    class_diagrams = {}
    print('    Skipped (gudhi not available) — H1 shape term will be 0.')


## Generate samples

Two methods:
- **Baseline** — random draws from the VAE prior $\mathcal{N}(0, I)$, decoded.
- **RL (sNES, shape-realness reward)** — REINFORCE search over the VAE latent with H1Shape · realness · balanced ambiguity as the reward. H1Shape requires features of both 2 and 6.


In [ ]:
N_GEN = 32

print('[A] Baseline — random VAE samples ...')
with torch.no_grad():
    baseline = vae.sample(N_GEN, device).cpu()

print('\n[B] RL / sNES with shape-realness reward ...')
rl_samples, rl_hist = generate_shape_creative_rl(
    vae, clf, ae, class_diagrams,
    warm_start_loader=loader_26,
    n_samples=N_GEN,
    n_episodes=80,
    lr_mu=0.05,
    lr_sigma=0.01,
    init_sigma=1.0,
    beta=0.01,
    weights=(1/3, 1/3, 1/3),
    device=device,
    warm_start_pairs=32,
)


## Score and visualise

Score both methods with the full shape-preserving metric, then show:
- Top-16 RL samples ranked by creativity.
- Baseline grid for comparison.
- RL training curves, including per-class H1 closeness (`close_2`, `close_6`) so you can see whether the policy is balancing features from both classes or collapsing onto one.
- Side-by-side box-plot comparison.


In [ ]:
print('Scoring baseline ...')
b_c, b_h1, b_r, b_a, b_close = score_shape_creativity(
    baseline, clf, ae, class_diagrams, device=device)
print('Scoring RL samples ...')
r_c, r_h1, r_r, r_a, r_close = score_shape_creativity(
    rl_samples, clf, ae, class_diagrams, device=device)

print('\nSummary — shape-preserving creativity metric')
hdr = (f"  {'Method':<12} {'Creativity':>10} {'H1Shape':>9} "
        f"{'close_2':>8} {'close_6':>8} {'Realness':>10} {'Ambiguity':>10}")
print(hdr); print('  ' + '-' * (len(hdr) - 2))
for name, c, h, clo, r, a in [
    ('Baseline', b_c, b_h1, b_close, b_r, b_a),
    ('RL (sNES)', r_c, r_h1, r_close, r_r, r_a),
]:
    c2 = clo.get(2, np.zeros(1)).mean() if clo else 0.0
    c6 = clo.get(6, np.zeros(1)).mean() if clo else 0.0
    print(f'  {name:<12} {c.mean():>10.4f} {h.mean():>9.4f} '
          f'{c2:>8.4f} {c6:>8.4f} {r.mean():>10.4f} {a.mean():>10.4f}')


In [ ]:
import os
os.makedirs('images', exist_ok=True)


def show_grid(images, creativity, h1, close_per, realness, ambiguity,
              title, filename, n: int = 16):
    n = min(n, len(images))
    order = np.argsort(creativity)[::-1][:n]
    cols  = 8
    rows  = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 2.4))
    axes = np.array(axes).flatten()
    c2_arr = close_per.get(2, np.zeros(len(images))) if close_per else np.zeros(len(images))
    c6_arr = close_per.get(6, np.zeros(len(images))) if close_per else np.zeros(len(images))
    for ax in axes:
        ax.axis('off')
    for i, idx in enumerate(order):
        axes[i].imshow(images[idx].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[i].set_title(
            f'C={creativity[idx]:.2f}\n'
            f'H1={h1[idx]:.2f} (c2={c2_arr[idx]:.2f} c6={c6_arr[idx]:.2f})\n'
            f'R={realness[idx]:.2f} A={ambiguity[idx]:.2f}',
            fontsize=6.0)
    plt.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved -> {filename}')


show_grid(rl_samples, r_c, r_h1, r_close, r_r, r_a,
          title='Shape-realness RL (sNES) — top 16 by creativity\n'
                '(C=Creativity  H1=min(c2,c6)  c2/c6=per-class H1 closeness  R=Realness  A=Ambiguity)',
          filename='images/shape_rl_top.png')

show_grid(baseline, b_c, b_h1, b_close, b_r, b_a,
          title='Baseline — random VAE samples (top 16 by creativity)\n'
                '(C=Creativity  H1=min(c2,c6)  c2/c6=per-class H1 closeness  R=Realness  A=Ambiguity)',
          filename='images/shape_baseline.png')


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6))

ax = axes[0]
ax.plot(rl_hist['reward'], color='#264653', linewidth=1.8)
ax.set_title('Creativity (reward)', fontsize=10)

ax = axes[1]
ax.plot(rl_hist['h1_shape'], color='#E76F51', linewidth=1.8, label='H1Shape = min(c2,c6)')
ax.plot(rl_hist['close_2'],  color='#4C72B0', linewidth=1.2, alpha=0.8, label='c2 (close to 2s)')
ax.plot(rl_hist['close_6'],  color='#DD8452', linewidth=1.2, alpha=0.8, label='c6 (close to 6s)')
ax.set_title('H1 shape realness', fontsize=10)
ax.legend(fontsize=7)

for ax, key, title, col in zip(
    axes[2:],
    ['realness', 'ambiguity', 'sigma'],
    ['Realness (AE)', 'Ambiguity (10-cls)', 'Policy sigma'],
    ['#2A9D8F', '#8172B2', '#888888'],
):
    ax.plot(rl_hist[key], color=col, linewidth=1.5)
    ax.set_title(title, fontsize=10)

for ax in axes:
    ax.set_xlabel('Episode', fontsize=9)
    ax.grid(True, alpha=0.3)
for ax in axes[:-1]:
    ax.set_ylim(-0.02, 1.02)

plt.suptitle('RL (sNES) training curves — shape-realness reward',
             fontsize=11, y=1.03)
plt.tight_layout()
plt.savefig('images/shape_rl_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('  Saved -> images/shape_rl_curves.png')


In [ ]:
from matplotlib.patches import Patch


def plot_shape_comparison(score_dict, filename='images/shape_comparison.png'):
    methods = list(score_dict.keys())
    labels  = ['Creativity', 'H1Shape', 'Realness', 'Ambiguity']
    colours = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    width   = 0.2
    x       = np.arange(len(methods))
    offsets = np.linspace(-(len(labels)-1)/2, (len(labels)-1)/2, len(labels)) * width

    fig, ax = plt.subplots(figsize=(8, 5))
    for k, (_, col) in enumerate(zip(labels, colours)):
        data = [score_dict[m][k] for m in methods]
        ax.boxplot(data, positions=x + offsets[k],
                   widths=width*0.85, patch_artist=True,
                   medianprops=dict(color='white', linewidth=2),
                   boxprops=dict(facecolor=col, alpha=0.7),
                   whiskerprops=dict(color=col),
                   capprops=dict(color=col),
                   flierprops=dict(marker='o', color=col, alpha=0.3, markersize=3))
    ax.set_xticks(x); ax.set_xticklabels(methods)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Shape-realness creativity — Baseline vs RL (sNES)')
    ax.legend(handles=[Patch(facecolor=c, alpha=0.7, label=l)
                        for l, c in zip(labels, colours)], fontsize=8)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved -> {filename}')


plot_shape_comparison({
    'Baseline':  (b_c, b_h1, b_r, b_a),
    'RL (sNES)': (r_c, r_h1, r_r, r_a),
})


## Advanced RL: Proximal Policy Optimisation (PPO)

REINFORCE / sNES does one gradient update per batch of decoded samples. Since our bottleneck is reward evaluation (H1 Wasserstein on 120 diagrams per sample), that's wasteful: we compute 32 expensive rewards, take *one* step, throw the batch away.

**PPO** (Schulman et al. 2017) extracts $K$ gradient updates from the same batch by treating the sampling distribution $\pi_{\text{old}}$ as fixed and re-weighting subsequent updates through an importance ratio $r_i(\theta) = \pi_\theta(z_i) / \pi_{\text{old}}(z_i)$. A *clip* on the ratio prevents the updated policy from drifting too far from the distribution that generated the samples — so the importance estimator stays valid:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_i \Big[\min\!\big(r_i(\theta)\,A_i,\ \mathrm{clip}(r_i(\theta),\,1-\varepsilon,\,1+\varepsilon)\,A_i\big)\Big]$$

with advantages $A_i$ z-scored within the batch for variance reduction. For our diagonal-Gaussian policy over $\mathbb{R}^d$ the log-probability is closed-form, so the whole objective is differentiable in $(\mu, \log\sigma)$ — we just call `.backward()` on it $K$ times.

**Why this is the right upgrade here:**
- *Sample efficiency per expensive reward.* $K$ updates per H1 evaluation vs 1 for REINFORCE.
- *Stability.* The clip replaces our hand-tuned EMA baseline and the one-shot natural-gradient scaling with a principled bound on per-step policy change.
- *Still a pure policy-gradient method.* No value network or trajectory machinery — our setting is a one-step bandit, and PPO degenerates cleanly to clipped importance-weighted Gaussian policy gradient in that regime.

We keep the warm-start, the KL prior to $\mathcal{N}(0, I)$ (as an auxiliary loss on the policy params), and the exact same shape-realness reward. The only thing that changes is the gradient estimator.


In [ ]:
def _gauss_logprob(z, mu, log_sigma):
    """Diagonal Gaussian log-prob.  All args torch tensors; returns (N,)."""
    var = torch.exp(2.0 * log_sigma)
    return -0.5 * (
        ((z - mu) ** 2 / var).sum(dim=-1)
        + 2.0 * log_sigma.sum(dim=-1)
        + mu.numel() * float(np.log(2.0 * np.pi))
    )


def generate_shape_creative_ppo(
    vae: VAE,
    clf: Classifier10,
    ae: GeneralAE,
    class_diagrams: dict,
    warm_start_loader,
    n_samples: int = 32,
    n_episodes: int = 80,
    n_epochs_per_batch: int = 4,
    clip_eps: float = 0.2,
    lr: float = 0.03,
    init_sigma: float = 1.0,
    beta_kl: float = 0.01,
    entropy_bonus: float = 1e-3,
    weights: tuple = (1/3, 1/3, 1/3),
    device: Union[str, torch.device] = 'cpu',
    warm_start_pairs: int = 32,
    seed: int = 0,
) -> tuple:
    """PPO (clipped-surrogate) latent-space search.

    Each episode:
      1. Sample N latent codes from frozen pi_old = N(mu, diag(sigma^2)).
      2. Decode, compute exact shape-realness reward (non-differentiable).
      3. Z-score advantages within the batch.
      4. For K epochs, minimise the clipped surrogate on the same batch:
           L = -E[ min( r * A, clip(r, 1-eps, 1+eps) * A ) ]
             + beta_kl * KL( pi_theta || N(0,I) )      -- trust region / prior
             - entropy_bonus * H[pi_theta]             -- keep exploration
         Gradients flow into (mu, log_sigma) via autograd.

    Same reward, warm-start, and prior as the REINFORCE version so the two
    runs are apples-to-apples.  Compute cost per episode is identical
    (one decode+H1 batch) but PPO performs K gradient steps against it.
    """
    assert TOPO_AVAILABLE, 'Install gudhi + persim for shape-preserving RL.'
    vae.eval(); clf.eval(); ae.eval()
    d = vae.latent_dim
    w1, w2, w3 = weights
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)

    mu0 = _warm_start_mu(vae, warm_start_loader, device, warm_start_pairs)
    print(f'  [PPO] warm-start mu from 2/6 centroid midpoint  ||mu||={np.linalg.norm(mu0):.3f}')
    mu        = nn.Parameter(torch.tensor(mu0, dtype=torch.float32, device=device))
    log_sigma = nn.Parameter(torch.full((d,), float(np.log(init_sigma)),
                                          dtype=torch.float32, device=device))
    opt = optim.Adam([mu, log_sigma], lr=lr)

    hist = {
        'reward': [], 'h1_shape': [], 'close_2': [], 'close_6': [],
        'realness': [], 'ambiguity': [], 'sigma': [],
        'ppo_loss': [], 'clip_frac': [],
    }

    for ep in range(n_episodes):
        # ── 1. Sample from frozen pi_old ──────────────────────
        with torch.no_grad():
            eps_t  = torch.tensor(rng.standard_normal((n_samples, d)),
                                   dtype=torch.float32, device=device)
            sigma_old  = torch.exp(log_sigma).detach()
            mu_old     = mu.detach().clone()
            log_sig_old = log_sigma.detach().clone()
            z      = mu_old[None] + sigma_old[None] * eps_t
            logp_old = _gauss_logprob(z, mu_old, log_sig_old)

            imgs     = vae.decoder(z)
            mse      = F.mse_loss(ae(imgs), imgs, reduction='none') \
                         .mean(dim=[1,2,3]).cpu().numpy()
            probs10  = clf.probs(imgs).cpu().numpy()
            imgs_cpu = imgs.cpu()

        realness            = 1.0 / (1.0 + mse)
        ambiguity           = balanced_ambiguity(probs10)
        h1_shape, close_per = h1_shape_realness(imgs_cpu, class_diagrams)
        rewards_np = (h1_shape ** w1) * (realness ** w2) * (ambiguity ** w3)

        # ── 2. Z-score advantages ────────────────────────────
        r_mean, r_std = float(rewards_np.mean()), float(rewards_np.std() + 1e-8)
        advantages = torch.tensor((rewards_np - r_mean) / r_std,
                                    dtype=torch.float32, device=device)

        # ── 3. K epochs of clipped surrogate on the same batch ──
        clip_fracs, ep_losses = [], []
        for _ in range(n_epochs_per_batch):
            logp_new = _gauss_logprob(z, mu, log_sigma)
            ratio    = torch.exp(logp_new - logp_old)

            unclipped = ratio * advantages
            clipped   = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
            policy_loss = -torch.min(unclipped, clipped).mean()

            # KL( N(mu, diag(sig^2)) || N(0, I) ) closed form
            var_theta = torch.exp(2.0 * log_sigma)
            kl_prior  = 0.5 * (var_theta.sum() + (mu ** 2).sum() - d
                                - (2.0 * log_sigma).sum())

            # Differential entropy of diagonal Gaussian (up to constants)
            entropy = log_sigma.sum()

            loss = policy_loss + beta_kl * kl_prior - entropy_bonus * entropy
            opt.zero_grad(); loss.backward(); opt.step()

            with torch.no_grad():
                frac = ((ratio - 1.0).abs() > clip_eps).float().mean().item()
                clip_fracs.append(frac)
                ep_losses.append(float(policy_loss.item()))

        sigma_now = torch.exp(log_sigma).detach().cpu().numpy()

        hist['reward'].append(float(rewards_np.mean()))
        hist['h1_shape'].append(float(h1_shape.mean()))
        hist['close_2'].append(float(close_per.get(2, np.zeros(1)).mean()))
        hist['close_6'].append(float(close_per.get(6, np.zeros(1)).mean()))
        hist['realness'].append(float(realness.mean()))
        hist['ambiguity'].append(float(ambiguity.mean()))
        hist['sigma'].append(float(sigma_now.mean()))
        hist['ppo_loss'].append(float(np.mean(ep_losses)))
        hist['clip_frac'].append(float(np.mean(clip_fracs)))

        if (ep + 1) % 10 == 0:
            c2 = close_per.get(2, np.zeros(1)).mean()
            c6 = close_per.get(6, np.zeros(1)).mean()
            print(f'  [PPO] ep {ep+1:>3}/{n_episodes}  '
                  f'R={rewards_np.mean():.4f}  '
                  f'H1={h1_shape.mean():.4f}  '
                  f'(c2={c2:.3f} c6={c6:.3f})  '
                  f'real={realness.mean():.4f}  '
                  f'amb={ambiguity.mean():.4f}  '
                  f'sigma={sigma_now.mean():.3f}  '
                  f'clip={np.mean(clip_fracs):.2f}')

    # Final sample from converged policy
    with torch.no_grad():
        sigma_fin = torch.exp(log_sigma)
        z_fin = mu[None] + sigma_fin[None] * torch.randn(n_samples, d, device=device)
        imgs_final = vae.decoder(z_fin).cpu()
    return imgs_final, hist


### Run PPO with matched hyperparameters

Same `n_samples`, `n_episodes`, `init_sigma`, `beta_kl`, and `warm_start_pairs` as the REINFORCE run, so the comparison isolates the gradient estimator. PPO adds `n_epochs_per_batch` (4) and `clip_eps` (0.2); reward-per-episode budget is identical.


In [ ]:
print('[C] PPO with shape-realness reward ...')
ppo_samples, ppo_hist = generate_shape_creative_ppo(
    vae, clf, ae, class_diagrams,
    warm_start_loader=loader_26,
    n_samples=N_GEN,
    n_episodes=80,
    n_epochs_per_batch=4,
    clip_eps=0.2,
    lr=0.03,
    init_sigma=1.0,
    beta_kl=0.01,
    entropy_bonus=1e-3,
    weights=(1/3, 1/3, 1/3),
    device=device,
    warm_start_pairs=32,
)

print('\nScoring PPO samples ...')
p_c, p_h1, p_r, p_a, p_close = score_shape_creativity(
    ppo_samples, clf, ae, class_diagrams, device=device)

print('\nSummary — shape-preserving creativity metric')
hdr = (f"  {'Method':<12} {'Creativity':>10} {'H1Shape':>9} "
        f"{'close_2':>8} {'close_6':>8} {'Realness':>10} {'Ambiguity':>10}")
print(hdr); print('  ' + '-' * (len(hdr) - 2))
for name, c, h, clo, r, a in [
    ('Baseline',   b_c, b_h1, b_close, b_r, b_a),
    ('REINFORCE',  r_c, r_h1, r_close, r_r, r_a),
    ('PPO',        p_c, p_h1, p_close, p_r, p_a),
]:
    c2 = clo.get(2, np.zeros(1)).mean() if clo else 0.0
    c6 = clo.get(6, np.zeros(1)).mean() if clo else 0.0
    print(f'  {name:<12} {c.mean():>10.4f} {h.mean():>9.4f} '
          f'{c2:>8.4f} {c6:>8.4f} {r.mean():>10.4f} {a.mean():>10.4f}')


### PPO sample grid


In [ ]:
show_grid(ppo_samples, p_c, p_h1, p_close, p_r, p_a,
          title='Shape-realness PPO — top 16 by creativity\n'
                '(C=Creativity  H1=min(c2,c6)  c2/c6=per-class H1 closeness  R=Realness  A=Ambiguity)',
          filename='images/shape_ppo_top.png')


### REINFORCE vs PPO convergence

Same x-axis (episode = one batch of reward evaluations) so the comparison is per-compute. PPO should reach higher rewards faster because it squeezes $K$ gradient steps out of each reward batch; the `clip_frac` panel shows how often the importance ratio hits the clip — if it's near 0 throughout, $K$ or `lr` could be increased.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
rl_col, ppo_col = '#264653', '#E76F51'

for ax, key, title in zip(
    axes[:4],
    ['reward',     'h1_shape',  'realness',  'ambiguity'],
    ['Creativity', 'H1Shape',   'Realness',  'Ambiguity'],
):
    ax.plot(rl_hist[key],  color=rl_col,  linewidth=1.7, label='REINFORCE / sNES')
    ax.plot(ppo_hist[key], color=ppo_col, linewidth=1.7, label='PPO')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Episode', fontsize=9)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

# PPO-only diagnostic: clip fraction
ax = axes[4]
ax.plot(ppo_hist['clip_frac'], color=ppo_col, linewidth=1.5)
ax.set_title('PPO clip fraction', fontsize=10)
ax.set_xlabel('Episode', fontsize=9)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

plt.suptitle('REINFORCE / sNES vs PPO — convergence under matched compute',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.savefig('images/reinforce_vs_ppo_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('  Saved -> images/reinforce_vs_ppo_curves.png')


### Final three-way distribution comparison


In [ ]:
plot_shape_comparison({
    'Baseline':   (b_c, b_h1, b_r, b_a),
    'REINFORCE':  (r_c, r_h1, r_r, r_a),
    'PPO':        (p_c, p_h1, p_r, p_a),
}, filename='images/shape_comparison_with_ppo.png')
